# CocoChorales-E: do the supplied audio and mistake labels agree?

This notebook separates three kinds of evidence:

1. **Dataset-wide MIDI census:** within-instrument overlap and the drift predicted by a sequential renderer, across explicitly named mirror splits.
2. **Specific audio examples:** unchanged author labels, supplied audio, and FluidSynth audio on one time axis; listening and spectral evidence do not depend on a model’s predicted mistakes.
3. **Cached model diagnostics:** a secondary check against the hypothesized sequential timeline, never replacement truth.

**No downloads, synthesis or model inference run by default.** Enable `RUN_CENSUS` to launch the parallel MIDI-only census. The listening cells use existing local audio. The benchmark and re-render controls are in [mistake.ipynb](mistake.ipynb).

The earlier investigation found specific timing mismatches in two violin stems. The census tests how broadly the prerequisite—overlapping notes within one instrument—occurs. It does **not** assume every overlap produces an audible annotation error. Report descriptive counts and denominators, not an untested statistical-significance claim.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "benchmarks/results"
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook

RUN_CENSUS = False
CENSUS_SPLITS = ("train", "valid", "test")
CENSUS_WORKERS = 24
CENSUS = RESULTS / "coco_e_audit_all_splits"
SOUNDFONT = RESULTS / "native_soundfont_2026-09-29"
NATIVE = RESULTS / "native_v5_min100_seed0_pieces3/coco"
manifest = json.loads((SOUNDFONT / "manifest.json").read_text())


## Why overlap breaks the sequential schedule

The author [generation script](https://github.com/ben2002chou/CocoChorales-E_MAESTRO-E/blob/main/create_cocochorales.py) calls MIDI-DDSP. Its [`note_list_to_sequence`](https://github.com/magenta/midi-ddsp/blob/main/midi_ddsp/utils/midi_synthesis_utils.py) converts notes into durations on a 4 ms grid, inserting rests only for positive gaps. Reconstructing cumulative durations serializes overlapping notes instead of retaining their absolute onsets.

For example, if G5 ends at 14.825 s but F♯5 starts at 14.275 s, the MIDI contains about 550 ms of overlap. Sequential rendering starts F♯5 near 14.824 s instead, delaying it and subsequent notes. The extra-note label can therefore describe a pitch that is absent at its labeled time in the supplied audio.

This is an inference from the source and the audio evidence below. The exact historical renderer revision/controls are unavailable. The hypothesis is not certified corrected ground truth.

## Dataset-wide prevalence — MIDI only, with explicit coverage

Download only `mistake/<piece>/mix.mid` from the pinned author-hosted mirror. Audit each nonempty instrument independently; concurrent quartet parts do not count as within-stem polyphony. Intervals are half-open: touching notes are not overlaps. The main prevalence threshold exceeds one 4 ms renderer frame; a separate column counts stems with overlaps over 100 ms.

The mix MIDI can have different tick precision from stem MIDI. The census measures structural overlap; the exact listening examples use the original stem MIDI/labels. `delayed_*` counts are predictions of the sequential schedule, not verified audio errors. All available selected pieces are attempted; failed downloads/parses are reported and must be resolved before claiming complete coverage. The mirror is a subset of the original 240,000-piece split list, and its equivalence to the original PolyTune release is unverified.

In [ ]:
if RUN_CENSUS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.modules.mistake.MistakeNotebook",
            str(CENSUS),
            "--splits",
            *CENSUS_SPLITS,
            "--workers",
            str(CENSUS_WORKERS),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Census disabled. Enable RUN_CENSUS to download/inspect MIDI in parallel.")


In [ ]:
if (CENSUS / "stems.csv").exists():
    coverage = json.loads((CENSUS / "metadata.json").read_text())
    stems = pd.read_csv(CENSUS / "stems.csv")
    display(
        pd.Series(
            {k: v for k, v in coverage.items() if k != "failures"}, name="Coverage"
        )
    )
    assert set(coverage["splits"]) == set(
        CENSUS_SPLITS
    ), "Cached scope differs from requested splits"
    assert not coverage["failures"], f"Incomplete census: {coverage['failures']}"
    assert coverage["selected_pieces"] == coverage["completed_pieces"]
    display(MistakeNotebook.audit_prevalence(stems).round(2))
    display(
        stems.assign(affected=stems.max_overlap_seconds.gt(0.004))
        .groupby(["split", "program"])
        .agg(stems=("piece", "size"), affected=("affected", "sum"))
    )
    display(stems.sort_values("max_overlap_seconds", ascending=False).head(20))
    ax = stems.groupby("split").max_overlap_seconds.plot.hist(
        bins=40, alpha=0.45, legend=True, figsize=(10, 3)
    )
else:
    print("No census at", CENSUS, "— run the preceding cell when ready.")


## Example 1 — `224962/0_violin`: an extra F♯5 arrives late

Listen from 13.2–16.3 s. The official correct G5 spans 13.577–14.825 s; the extra F♯5 spans 14.275–15.005 s. Listen for whether both pitches coexist at 14.4–14.65 s, and then for the following G♯5 around 15.0 s. The dashed schedule is a hypothesis; the solid colored bars are the supplied labels.

The spectrograms show the same time/frequency window with label fundamentals overlaid in cyan. Each player normalizes its own clip for audibility, so compare timing and pitch coexistence, not absolute loudness.

In [ ]:
clips = MistakeNotebook.audit_example(manifest, "224962/0_violin", 13.2, 16.3)
display(
    MistakeNotebook.audit_spectral_evidence(
        manifest, "224962/0_violin", [(14.4, 14.65, 79, 78), (14.95, 15.15, 78, 80)]
    ).round(2)
)


These measurements compare Hann-windowed FFT peaks within ±30 cents of the candidate fundamentals (65,536-point FFT). The prior saved investigation found F♯5 about 44 dB below G5 in the supplied first window, versus nearly equal peaks in FluidSynth. The following G♯5 likewise appeared late. Recompute above and listen to assess the evidence yourself.

Narrow-band peaks are not calibrated loudness or a full transcription. Their role is to corroborate the missing/delayed labeled pitch independently of Attune. The problem is temporal supervision: a real extra pitch can be present later while its supplied timestamp is wrong.

## Example 2 — `224962/1_violin`: a separate 216 ms overlap

A4 spans 7.455–8.055 s while C♯5 starts at 7.839 s. Listen for the start of C♯5 and the following notes. This is a separate stem but the same piece, not an independent dataset-wide prevalence estimate.

In [ ]:
clips = MistakeNotebook.audit_example(manifest, "224962/1_violin", 7.1, 10.0)
display(
    MistakeNotebook.audit_spectral_evidence(
        manifest, "224962/1_violin", [(7.88, 8.02, 69, 73)]
    ).round(2)
)


## Secondary check — saved detections against two timelines

This only reads previously saved Attune events. Compare all 12 stems at the same onset gate, including unaffected stems. The sequential-schedule row is a diagnostic counterfactual, not new labels or a corrected benchmark score. Offset accuracy and actual human transcription are not established by onset-only F1.

In [ ]:
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook

summary, details, boundaries = MistakeNotebook.renderer_inspect(NATIVE, SOUNDFONT)
display(summary)
display(
    details[details.gate.eq(0.1)].pivot(
        index="case", columns=["renderer", "reference"], values="fn"
    )
)


## What the evidence supports

Use the census’s actual scope and denominators when reporting how often within-stem polyphony occurs. Use the audio examples to demonstrate that the supplied labels and rendered timeline can disagree; do not convert the MIDI-risk count into a count of verified acoustic failures.

The two examples support investigating noisy timing supervision wherever the same rendering path received overlapping input. They do not prove that every overlapping performance was misrendered, that every training example was consumed, or that a checkpoint learned the artifact. A human listening audit over a prespecified sample from the census would be the next step for estimating acoustic-error prevalence.

FluidSynth preserves simultaneous MIDI events, making it useful for the separate E benchmark. It also changes timbre/attacks and retains polyphony; model scores on it measure a changed audio domain. Keep original and re-rendered results separate.

[Detailed prior evidence](../../docs/archive/native-coco-renderer-timing-2026-09-29.md) · [Preserved promoted-pipeline audit](archive/mistake_promoted_audit_2026-09-27.ipynb)